# Outline→Blog Post Generator -- notebook demo

This notebook is a runnable demo of the **Outline→Blog Post Generator** project from the course: [`docs/projects/blog-post-generator/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/blog-post-generator), companion to the fuller local CLI at [`examples/blog-post-generator/generate.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/blog-post-generator/generate.py).

It reads a rough Markdown outline, hands it to a free-tier LLM with a faithfulness-focused expansion prompt, and prints back a polished blog post draft.

## A note on running this in a notebook

The real version of this tool (`examples/blog-post-generator/generate.py`) reads **your own outlines** from files on disk -- that's the whole point of the tool. Colab, Kaggle, and Binder don't have your files.

So **this demo adapts the tool**: it fetches the bundled sample outline (`sample_outline.md`) straight from the course repo. That runs every piece of the tool (the file reading, the system prompt, the LLM call, the structured output) honestly -- it's just not pointed at your real outline. **Locally, or in a GitHub Codespace, you'd point it at your own files instead** -- see the [project walkthrough](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/blog-post-generator) for that path.

In [ ]:
!pip install -q openai

## Get the sample outline

Fetch the bundled `sample_outline.md` straight from the course repo -- a substantive outline about building a habit-tracking heatmap, with a real thesis, a story arc, and a deliberate `TODO` bullet to see how the model handles honest placeholders.

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/examples/blog-post-generator/sample_outline.md"
outline = urllib.request.urlopen(url).read().decode("utf-8")
print(f"Loaded sample outline: {len(outline)} characters")
print(outline)

## The expansion system prompt

This is the exact `SYSTEM_PROMPT` from `generate.py` -- it's what turns a general-purpose chat model into a disciplined outline-expander: cover every section in order, never invent sections or examples, and mark outline holes with honest `[expand: ...]` notes instead of fabricating.

In [ ]:
SYSTEM_PROMPT = """\
You are an experienced, clear-writing blog post editor who expands outlines
into prose.

You will be given a Markdown outline. Expand it into a complete, well-
structured blog post draft. Follow these rules:

- Faithfulness: Cover every section and bullet in the outline, in the order
  given. NEVER invent sections, claims, or examples that are not in the
  outline. If a bullet is a question or a placeholder ("TODO", "need an
  example here"), write it as an honest rough passage and mark it with a
  bracketed note like [expand: find a concrete example], rather than
  inventing something to fill it.
- Structure: Preserve the outline's headings (##, ###) as your section
  headings. Add an engaging intro paragraph after the title, and a short
  conclusion, IF the outline calls for them -- but do not add sections the
  outline doesn't imply.
- Prose: Write in clear, conversational but professional prose. Expand each
  bullet into one or more paragraphs. Do not pad with fluff, repetition, or
  generic filler sentences.
- Voice: Write in the first person, in a confident but plain voice, as if
  the outline's author were writing it.

Output ONLY the draft. No preamble, no "here is your draft", no commentary.
"""

## Get a free-tier API key

This demo defaults to **GitHub Models** -- free, no separate signup, just a personal access token with the `models: read` scope from [github.com/settings/tokens](https://github.com/settings/tokens). Any of the other five providers wired up in `generate.py` (Gemini, Groq, Mistral, Cerebras, OpenRouter) work too -- see that file's `PROVIDERS` dict for their base URLs and env var names, and adjust `LLM_PROVIDER` below.

The key is entered with `getpass` so it never gets typed into a visible cell or saved into this notebook's output -- never hardcode a real API key here.

In [ ]:
import os
from getpass import getpass

LLM_PROVIDER = "github"  # change to gemini / groq / mistral / cerebras / openrouter if you prefer
os.environ["GITHUB_TOKEN"] = getpass("Enter your free-tier GitHub Models token (GITHUB_TOKEN): ")

## The generation logic itself

This mirrors `truncate`, `PROVIDERS`, and `generate` from `generate.py` directly -- the same truncation cap, the same free-tier providers (all exposed through the `openai` client, just pointed at each provider's own OpenAI-compatible endpoint), and the same call shape.

In [ ]:
from openai import OpenAI

MAX_OUTLINE_CHARS = 12_000


def truncate(outline: str, max_chars: int = MAX_OUTLINE_CHARS) -> str:
    """Cuts an oversized outline down to a size that fits a free-tier context window."""
    if len(outline) <= max_chars:
        return outline
    return outline[:max_chars] + f"\n\n... [outline truncated -- {len(outline) - max_chars} more characters not shown] ..."


def _build_github_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.github.ai/inference")


def _build_gemini_client() -> OpenAI:
    return OpenAI(
        api_key=os.environ["GOOGLE_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )


def _build_groq_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")


def _build_mistral_client() -> OpenAI:
    return OpenAI(api_key=os.environ["MISTRAL_API_KEY"], base_url="https://api.mistral.ai/v1")


def _build_cerebras_client() -> OpenAI:
    return OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1")


def _build_openrouter_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")


PROVIDERS = {
    "github": (_build_github_client, "gpt-4o-mini"),
    "gemini": (_build_gemini_client, "gemini-3.5-flash"),
    "groq": (_build_groq_client, "llama-3.3-70b-versatile"),
    "mistral": (_build_mistral_client, "mistral-small-latest"),
    "cerebras": (_build_cerebras_client, "llama-3.3-70b"),
    "openrouter": (_build_openrouter_client, "meta-llama/llama-3.3-70b-instruct:free"),
}


def generate(outline: str, provider: str = LLM_PROVIDER) -> str:
    """Sends an outline to a free-tier LLM and returns the expanded blog post draft."""
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    build_client, model = PROVIDERS[provider]
    client = build_client()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Here is my outline:\n\n```markdown\n{truncate(outline)}\n```"},
        ],
    )
    return response.choices[0].message.content

## Run the generation

Expanding the bundled sample outline above into a full draft. Read the result critically: does it cover every section? Did it invent anything? Did it handle the `TODO` bullet honestly? Then edit it until it's *yours* -- that editing step is the actual writing this tool is a starting point for.

In [ ]:
print(f"Expanding a {len(outline)}-char outline into a blog post draft...\n")
draft = generate(outline)
print(draft)